# Ingestion Pipeline — COSORA

Ingesta unificada: parser `python-docx` + tablas, chunking por frases, embeddings **MrBERT-es** o **E5** (configurable), ChromaDB en Drive o local.

**Prerequisito query:** ejecutar este notebook antes de `query_pipeline.ipynb`.

| Variable | Valores |
|----------|--------|
| `RUNTIME` | `auto` (detectado), `colab`, `local` |
| `EMBED_BACKEND` | `mrbert` \| `e5` |
| `INDEX_BOTH` | `True` indexa ambas colecciones (para benchmark) |

> **Nota:** Este notebook es auto-contenido. Funciona tanto en local como en Colab sin necesidad de subir archivos `.py` adicionales.

## 0. Setup y configuración

Instalación de dependencias e importaciones. La auto-detección `RUNTIME='auto'` detecta si estamos en Colab o en local.

In [5]:
%pip install -q python-docx chromadb sentence-transformers transformers torch rank_bm25

# Colab: descomentar si usas .doc legacy
!apt-get install -y antiword > /dev/null 2>&1

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-DETECT: Colab vs Local
# ═══════════════════════════════════════════════════════════════════════
import glob
import os
import re
import subprocess
import sys
import unicodedata
from pathlib import Path
from typing import Any

import chromadb
import torch
from docx import Document
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

IN_COLAB = "google.colab" in sys.modules

# ─── Configuración ────────────────────────────────────────────────────
RUNTIME = "auto"           # "auto" | "colab" | "local"
INDEX_BOTH = True          # True → indexar mrbert + e5 (benchmark)
EMBED_BACKEND = "mrbert"   # "mrbert" | "e5" (solo si INDEX_BOTH=False)

if RUNTIME == "auto":
    RUNTIME = "colab" if IN_COLAB else "local"

if RUNTIME == "colab" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ─── Rutas ────────────────────────────────────────────────────────────
def resolve_paths(runtime, chroma_path_override=None):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = chroma_path_override or f"{docs_dir}/chroma_db"
    else:
        # Local: notebooks/ está en el mismo dir que este notebook
        nb_dir = Path(".").resolve()
        # Si estamos en notebooks/, subimos un nivel para el root del proyecto
        if nb_dir.name == "notebooks":
            root = nb_dir.parent
        else:
            root = nb_dir
        docs_dir = str(root / "data" / "raw")
        chroma_path = chroma_path_override or str(nb_dir / "chroma_db")
    return docs_dir, chroma_path

DOCS_DIR, CHROMA_PATH = resolve_paths(RUNTIME)

print(f"IN_COLAB={IN_COLAB}  RUNTIME={RUNTIME}")
print(f"DOCS_DIR={DOCS_DIR}")
print(f"CHROMA_PATH={CHROMA_PATH}")
print(f"INDEX_BOTH={INDEX_BOTH}")
if not INDEX_BOTH:
    print(f"EMBED_BACKEND={EMBED_BACKEND}")

# ═══════════════════════════════════════════════════════════════════════
# COSORA SHARED — código embebido (no requiere cosora_shared.py)
# ═══════════════════════════════════════════════════════════════════════

EMBED_BACKENDS: dict[str, dict[str, Any]] = {
    "mrbert": {
        "model_id": "BSC-LT/MrBERT-es",
        "collection": "cosora_chunks_mrbert",
        "type": "transformers_cls",
        "query_prefix": "",
        "doc_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "type": "sentence_transformers",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}


def backend_config(backend):
    if backend not in EMBED_BACKENDS:
        raise ValueError(f"EMBED_BACKEND desconocido: {backend}. Usa: {list(EMBED_BACKENDS)}")
    return EMBED_BACKENDS[backend]


def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"[\u00ad\u2010-\u2015\u2212\uff0d]", "-", text)
    text = re.sub(r"[\u2018\u2019\u201a\u201b]", "'", text)
    text = re.sub(r"[\u201c\u201d\u201e\u201f]", '"', text)
    text = re.sub(r"[\u2022\u2023\u25cf\u25e6\u2043\u2219\u00b7]", "-", text)
    text = re.sub(r"(?m)^\s*\d{1,4}\s*$", "", text)
    text = re.sub(r"([.\-_=])\1{2,}", r"\1", text)
    text = re.sub(r"\s+([.,;:!?)])", r"\1", text)
    lines = [line for line in text.split("\n") if line.count("|") < 3]
    text = "\n".join(lines)
    text = re.sub(
        r"(?mi)^\s*(fecha|lugar|hora de inicio|hora de finalizacion|asistentes)\s*$",
        "",
        text,
    )
    text = re.sub(r"(?mi)^\s*(firmado|firma).*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_bad_chunk(chunk, min_words=40, min_alpha_ratio=0.65):
    words = len(chunk.split())
    if words < min_words:
        return True
    alpha_ratio = sum(c.isalpha() for c in chunk) / max(len(chunk), 1)
    return alpha_ratio < min_alpha_ratio


def chunk_text(text, chunk_size=500, overlap=100, min_chunk_size=50):
    chunks = []
    if not text:
        return chunks
    stride = chunk_size - overlap
    start = 0
    text_len = len(text)
    while start < text_len:
        end = min(start + chunk_size, text_len)
        if end < text_len:
            search_from = start + int(chunk_size * 0.8)
            best = -1
            for punct in ".!?;":
                pos = text.rfind(punct, search_from, end)
                if pos > best:
                    best = pos
            if best != -1:
                end = best + 1
            else:
                space = text.rfind(" ", search_from, end)
                if space != -1:
                    end = space
        chunk = text[start:end].strip()
        if len(chunk) >= min_chunk_size and not is_bad_chunk(chunk):
            chunks.append(chunk)
        start += stride
    return chunks


def extract_docx(filepath):
    doc = Document(filepath)
    texts = []
    for paragraph in doc.paragraphs:
        text = normalize_text(paragraph.text)
        if text:
            texts.append(text)
    for table in doc.tables:
        for row in table.rows:
            row_texts = []
            seen = set()
            for cell in row.cells:
                text = normalize_text(cell.text)
                if not text or text in seen:
                    continue
                seen.add(text)
                row_texts.append(text)
            if row_texts:
                texts.append(" || ".join(row_texts))
    combined = "\n".join(texts).strip()
    return combined or None


def extract_doc(filepath):
    result = subprocess.run(
        ["antiword", filepath],
        capture_output=True, text=True, check=False,
    )
    if result.returncode == 0 and result.stdout.strip():
        return result.stdout
    return None


def read_document(filepath):
    lower = filepath.lower()
    if lower.endswith(".docx"):
        return extract_docx(filepath)
    if lower.endswith(".doc"):
        return extract_doc(filepath)
    return None


def list_document_paths(docs_dir):
    paths = []
    paths.extend(glob.glob(os.path.join(docs_dir, "*.docx")))
    paths.extend(glob.glob(os.path.join(docs_dir, "*.doc")))
    return sorted(paths)


def process_documents(docs_dir):
    documents = []
    for filepath in list_document_paths(docs_dir):
        filename = os.path.basename(filepath)
        doc_id = os.path.splitext(filename)[0]
        raw = read_document(filepath)
        if not raw:
            continue
        cleaned = clean_text(raw)
        chunks = chunk_text(cleaned)
        if not chunks:
            continue
        documents.append({"doc_id": doc_id, "chunks": chunks, "filepath": filepath})
    return documents


class Embedder:
    def __init__(self, backend):
        self.backend = backend
        self.cfg = backend_config(backend)
        self._st_model = None
        self._model = None
        self._tokenizer = None

    def _load(self):
        if self.cfg["type"] == "sentence_transformers":
            if self._st_model is None:
                from sentence_transformers import SentenceTransformer
                self._st_model = SentenceTransformer(self.cfg["model_id"])
            return
        if self._model is None:
            self._tokenizer = AutoTokenizer.from_pretrained(self.cfg["model_id"])
            self._model = AutoModel.from_pretrained(self.cfg["model_id"])
            self._model.eval()

    def embed_one(self, text, *, is_query=False):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            vec = self._st_model.encode(prefix + text)
            return vec.tolist() if hasattr(vec, "tolist") else list(vec)
        assert self._tokenizer is not None and self._model is not None
        inputs = self._tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self._model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    def embed_batch(self, texts, *, is_query=False, batch_size=16):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            prefixed = [prefix + t for t in texts]
            vecs = self._st_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True)
            return vecs.tolist()
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            for text in batch:
                all_embeddings.append(self.embed_one(text, is_query=is_query))
        return all_embeddings


def index_documents(documents, backend, chroma_path, batch_size=32):
    cfg = backend_config(backend)
    embedder = Embedder(backend)
    all_texts, all_doc_ids, all_ids = [], [], []
    for doc in documents:
        for i, chunk in enumerate(doc["chunks"]):
            all_texts.append(chunk)
            all_doc_ids.append(doc["doc_id"])
            all_ids.append(f"{doc['doc_id']}__c{i:04d}")
    if not all_texts:
        return 0
    embeddings = embedder.embed_batch(all_texts, batch_size=batch_size)
    client = chromadb.PersistentClient(path=chroma_path)
    try:
        client.delete_collection(cfg["collection"])
    except Exception:
        pass
    collection = client.create_collection(
        name=cfg["collection"],
        metadata={"hnsw:space": "cosine", "embed_backend": backend},
    )
    metadatas = [
        {"doc_id": doc_id, "chunk_id": cid, "embed_backend": backend}
        for doc_id, cid in zip(all_doc_ids, all_ids)
    ]
    for i in range(0, len(all_texts), 5000):
        j = min(i + 5000, len(all_texts))
        collection.add(
            ids=all_ids[i:j],
            documents=all_texts[i:j],
            embeddings=embeddings[i:j],
            metadatas=metadatas[i:j],
        )
    return collection.count()


def get_chroma_collection(chroma_path, backend):
    cfg = backend_config(backend)
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection(cfg["collection"]), cfg


def dense_search(collection, embedder, query, k=100):
    q_vec = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]

print("\n✅ Código COSORA cargado correctamente")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  RUNTIME=colab
DOCS_DIR=/content/drive/MyDrive/RAG_UPC_Final_project
CHROMA_PATH=/content/drive/MyDrive/RAG_UPC_Final_project/chroma_db
INDEX_BOTH=True

✅ Código COSORA cargado correctamente


## 1. Procesar documentos (parser + chunking)

In [ ]:
if not os.path.isdir(DOCS_DIR):
    raise FileNotFoundError(
        f"No existe DOCS_DIR: {DOCS_DIR}. "
        "Coloca los .docx/.doc en data/raw (local) o en Drive (colab)."
    )

documents = process_documents(DOCS_DIR)
total_chunks = sum(len(d["chunks"]) for d in documents)
print(f"{len(documents)} documentos, {total_chunks} chunks")
for doc in documents[:5]:
    print(f"  {doc['doc_id']}: {len(doc['chunks'])} chunks")


31 documentos, 582 chunks
  244170-DOB-AVO-05-V00-A0: 4 chunks
  244170-DOB-AVO-07-V00-A0: 5 chunks
  244170-DOB-AVO-09-V00: 4 chunks
  244170-DOB-AVO-10-V00-A0: 4 chunks
  244170-DOB-AVO-11-V00: 4 chunks
  ...


## 2. Indexar en ChromaDB

In [8]:
backends_to_index = list(EMBED_BACKENDS.keys()) if INDEX_BOTH else [EMBED_BACKEND]

for backend in backends_to_index:
    cfg = backend_config(backend)
    print(f"\n--- Indexando backend={backend} → colección {cfg['collection']} ---")
    n = index_documents(documents, backend, CHROMA_PATH, batch_size=16)
    print(f"ChromaDB: {n} chunks en {CHROMA_PATH}")


--- Indexando backend=mrbert → colección cosora_chunks_mrbert ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.83M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

ChromaDB: 582 chunks en /content/drive/MyDrive/RAG_UPC_Final_project/chroma_db


## 3. Verificación

In [9]:
test_query = "¿Qué se decidió sobre el talud?"

backends_to_test = list(EMBED_BACKENDS.keys()) if INDEX_BOTH else [EMBED_BACKEND]

for backend in backends_to_test:
    collection, cfg = get_chroma_collection(CHROMA_PATH, backend)
    embedder = Embedder(backend)
    hits = dense_search(collection, embedder, test_query, k=3)

    print("\n" + "=" * 70)
    print(f"Backend: {backend}")
    print(f"Colección: {cfg['collection']} ({collection.count()} chunks)")
    print(f"Query prueba: {test_query}\n")
    for i, h in enumerate(hits, 1):
        print(f"{i}. [{h['meta']['doc_id']}] {h['text'][:120]}...")
print("\nOK: verificación completada")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Backend: mrbert
Colección: cosora_chunks_mrbert (582 chunks)
Query prueba: ¿Qué se decidió sobre el talud?

1. [254275-DO-AVO-28-V01] Principales asistentes: FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (direcc...
2. [254275-DO-AVO-14-V07] Principales asistentes: FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (direcc...
3. [254275-DO-AVO-15-V07] Principales asistentes: FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (direcc...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Backend: e5
Colección: cosora_chunks_e5 (582 chunks)
Query prueba: ¿Qué se decidió sobre el talud?

1. [254275-DO-AVO-15-V07] adas en los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud || Posterior a la reunión de obra, DF envía ...
2. [254275-DO-AVO-16-V07] iará el certificado a la DF. AR-21 Plano camino provisional || DF solicita a la UTE que aporte la documentación gráfica ...
3. [254275-DO-AVO-14-V07] envía en resultado de la verificación de la documentación aportada por la UTE sobre la estabilidad del talud, solicitand...

OK: verificación completada
